# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

#### Although the starter notebook uses classification models for demonstration, the business objective of this lane is best described as a ranking/scoring task. The goal is not simply to classify pages as "refresh" or "do not refresh," but to assign each page a score so that SEO and content teams can prioritize which pages to review first. The output is therefore an ordered list of pages ranked by their estimated review priority. A classification model may be one component of the solution, but the final business deliverable is a ranked queue of recommendations.

In [5]:
import pandas as pd

df = pd.read_csv("C:/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")   
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

#### The ideal target would be a manually reviewed label indicating whether a page should be refreshed. Since that label is unavailable in the starter dataset, a proxy target must be used. The dataset provides fields such as trend_direction and trend_pct, which describe recent changes in page performance. These variables can serve as proxies for identifying pages whose search performance is declining. During later iterations, additional business rules or editorial feedback could be incorporated to create higher-quality labels.

In [6]:
df["trend_direction"].value_counts()

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

In [7]:
df[["trend_direction", "trend_pct"]].head()

,trend_direction,trend_pct
0,down,-41.4
1,down,-57.7
2,down,-60.9
3,stable,-13.8
4,down,-34.7


## 3. Success metric

*One metric you can defend. What number means 'good'?*

#### The success of this task should be measured using Precision@K, where K represents the number of pages the content team can realistically review during a planning cycle. Precision@K directly measures how many of the highest-ranked recommendations are genuinely valuable review candidates. This metric aligns better with the business objective than overall accuracy because the organization only acts on the highest-ranked pages.

In [8]:
print("Example business metric:")
print("Precision@20")
print("Precision@50")

Example business metric:
Precision@20
Precision@50


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

#### The unit of analysis for this project is one content page. Each row in the dataset represents a single page together with its search visibility, engagement, freshness, content characteristics, and historical performance metrics. The model therefore makes one recommendation for each page, allowing the final output to be a ranked list of pages for editorial review.

In [12]:
print("Dataset Row: ",df.shape[0],"Col: ",df.shape[1])
print("Columns: ",df.columns.tolist())
df.head()

Dataset Row:  30000 Col:  44
Columns:  ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

#### A simple rule-based system might recommend refreshing every page older than 180 days or every page with declining impressions. While these rules are easy to understand, they ignore interactions between multiple signals. Search performance depends on a combination of factors including visibility, engagement, freshness, content type, competition, and historical trends. Machine learning can learn patterns across many features simultaneously and produce more accurate rankings than a single manually designed rule. This was already demonstrated during Week 1, where a Decision Tree improved Precision@50 over the hand-crafted baseline.

In [13]:
features = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
]

df[features].head()

,impressions_90d,avg_position,ctr,engagement_rate,content_age_days,days_since_last_update
0,3803,10.6,0.76,5.88,187,20
1,15320,20.3,0.05,0.00,445,25
2,12581,36.5,0.09,0.00,141,20
3,11751,6.2,0.49,1.28,463,22
4,19140,44.0,0.13,0.00,263,14


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.